# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We will download the dataset metadata from its Croissant JSON-LD schema, inspect available record sets and fields, load the tabular data, and perform basic exploratory data analysis, referencing entities by their `@id` as per the Croissant specification.

### Dataset Source
The dataset schema is at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published on: {getattr(meta, 'datePublished', None)}")
print(f"Authors: {getattr(meta, 'author', None)}")

## 2. Data Overview

Review available record sets, fields, and their `@id`. This helps us understand what data tables are included in the dataset and their structures.

In [ ]:
# List record sets and their fields
print("Available Record Sets:\n----------------------")
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    else:
        print("  No fields defined.")
    print()

# For illustrative purposes, show up to 3 sample records from the first record set:
if len(dataset.metadata.record_sets) > 0:
    record_set_id = dataset.metadata.record_sets[0].id
    print(f"\nSample records from first RecordSet (@id={record_set_id}):")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        if i >= 3:
            break
        print(rec)


## 3. Data Extraction

Let's load data from all the record sets of the dataset into Pandas DataFrames. We continue to reference record sets and fields by their `@id` according to the Croissant schema.

In [ ]:
# Extract all record set data as DataFrames, keyed by record set @id
dataframes = {}
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    # Load all records from the record set
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found in RecordSet: {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded RecordSet @id: {record_set_id}, shape: {df.shape}")
    print(f"Columns: {list(df.columns)}\n")

# Pick a record set for analysis (choose first one as an example)
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Example head of DataFrame for RecordSet @id: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply some common data processing operations using a numeric field from the chosen record set. We filter records by a threshold on a numeric field, normalize it, and show grouping by a categorical field (if available). All fields are referenced by their `@id`.

In [ ]:
import numpy as np
# Select first valid numeric field for illustration
rs = [x for x in dataset.metadata.record_sets if x.id == selected_record_set_id][0]
numeric_field_id = None
for field in rs.fields:
    # Common numeric types: Integer, Float, Number
    dt = getattr(field, 'data_type', '')
    if dt and dt.lower() in ('float', 'integer', 'number'):
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    print('No numeric field found to analyze.')
else:
    print(f"Using numeric field '@id'={numeric_field_id}")

    # Ensure the field is present and coerce to numeric
    df = dataframes[selected_record_set_id]
    if numeric_field_id not in df:
        print(f"Field {numeric_field_id} not found in DataFrame columns.")
    else:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Use mean as threshold example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)}");
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a suitable categorical field
        group_field = None
        for field in rs.fields:
            dt = getattr(field, 'data_type', '')
            if (dt.lower() in ('text', 'string')) and (field.id != numeric_field_id):
                group_field = field.id
                break
        if group_field and group_field in df:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field}' (mean of {numeric_field_id}):")
            display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field (if available), and the relationship grouped by the chosen categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df:
    # Histogram
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping above succeeded, show bar plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to use the `mlcroissant` library to load and explore a Croissant-formatted biomedical dataset via its JSON-LD schema. We performed metadata inspection, loaded all available record sets by their `@id`, selected and analyzed a numeric field, and visualized basic patterns in the data. All references to entities such as record sets and fields were made by their Croissant `@id` in accordance with best FAIR practices. For advanced analysis, researchers are encouraged to further explore the semantic structure and provenance links provided in the Croissant metadata.